# RAG assignment: baseline retrieval vs cross-encoder re-ranking (D&D 5e SRD, Qwen3.5-9B, 25 questions)

One notebook, run top to bottom. It implements the two retrieval workflows from the assignment over the **same chunked corpus**
and compares the answers a local **Qwen3.5-9B** gives for 25 questions.

**Flow**

1. Load the PDF and cut it into overlapping chunks (sections 1).
2. Define 25 test questions and, for each, the passage of the SRD that answers it, chosen by hand (section 2).
3. **Pipeline 1, baseline:** a bi-encoder embeds every chunk into a FAISS inner-product index; for each question the top 10 chunks are retrieved and the first 3 go to the LLM (section 3).
4. **Pipeline 2, re-ranking:** the same 10 candidates are scored jointly with a cross-encoder, re-sorted, and the first 3 go to the LLM (section 3).
5. Retrieval metrics, rank shifts and timings (sections 4 to 6).
6. Qwen3.5-9B answers every question twice with an identical prompt: Answer 1 from the baseline top 3, Answer 2 from the re-ranked top 3 (sections 7 and 8).
7. Answers are graded against gold key facts and the pipelines are compared, with a latency table (sections 9 and 10).

**Models and where they are used**

| Role | Model | Where |
|---|---|---|
| Bi-encoder (stage 1) | `BAAI/bge-small-en-v1.5` | section 3, `EMBEDDERS["bge-small"]` |
| Bi-encoder (stage 1) | `google/embeddinggemma-2` (float32) | section 3, `EMBEDDERS["embeddinggemma-2"]` |
| Cross-encoder (stage 2) | `cross-encoder/ms-marco-MiniLM-L-6-v2` | section 3, `ce = CrossEncoder(...)` |
| LLM (answers) | `Qwen/Qwen3.5-9B`, fp16, thinking off, split across two 11 GB GPUs | section 7 |

The two embedders are run side by side so the effect of re-ranking can be seen for two different first-stage models; everything else is identical.

**Needs:** two GPUs of about 11 GB (the 9B model is about 18 GB in fp16), the SRD PDF (`data/SRD_CC_v5.1.pdf`), and the three models in the Hugging Face cache.
Paths can be set with the environment variables `RAG_PDF`, `RAG_OUT` and `MODEL_DIR` (the cluster run used a local copy of the model on node-local disk).

In [1]:
import os, re, json, time, gc, hashlib, textwrap
import numpy as np, pandas as pd, faiss, pymupdf, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

PDF = os.environ.get("RAG_PDF", "data/SRD_CC_v5.1.pdf")
OUT = os.environ.get("RAG_OUT", "outputs")
LLM_NAME = os.environ.get("MODEL_DIR", "Qwen/Qwen3.5-9B")
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda"
assert torch.cuda.device_count() >= 2, "the 9B model needs two GPUs of about 11 GB"
print("gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

gpus: ['NVIDIA GeForce RTX 2080 Ti', 'NVIDIA GeForce RTX 2080 Ti']


## 1. Load the PDF and chunk it

The SRD is laid out in two columns. Reading a page in plain order interleaves the columns and produces garbled sentences,
so each page is cut into a left and a right half and each half is read separately. A 40 pt strip at the top and bottom
(running header and footer) is dropped. The words are then split into windows of 200 words with a 40-word overlap, so a
rule that straddles a chunk boundary still appears whole in at least one chunk.

In [2]:
CHUNK_WORDS, OVERLAP = 200, 40

doc = pymupdf.open(PDF)
words, word_page = [], []
for pno, page in enumerate(doc, 1):
    r = page.rect
    top, bot, mid = r.y0 + 40, r.y1 - 40, (r.x0 + r.x1) / 2
    for col in (pymupdf.Rect(r.x0, top, mid, bot), pymupdf.Rect(mid, top, r.x1, bot)):   # left half, then right half
        for w in page.get_text("text", clip=col).split():
            words.append(w); word_page.append(pno)

chunks = [{"id": i, "page": word_page[s], "text": " ".join(words[s:s + CHUNK_WORDS])}
          for i, s in enumerate(range(0, len(words), CHUNK_WORDS - OVERLAP))]
print(f"{len(doc)} pages, {len(words)} words -> {len(chunks)} chunks")
print("\nSample chunk:\n", textwrap.fill(chunks[300]["text"], 110))

403 pages, 240639 words -> 1504 chunks

Sample chunk:
 clear on the jump costs a foot of movement. In some circumstances, your GM might allow you to make a Strength
(Athletics) check to jump higher than you normally can. You can extend your arms half your height above
yourself during the jump. Thus, you can reach above you a distance equal to the height of the jump plus 1½
times your height. The Environment By its nature, adventuring involves delving into places that are dark,
dangerous, and full of mysteries to be explored. The rules in this section cover some of the most important
ways in which adventurers interact with the environment in such places. Falling A fall from a great height is
one of the most common hazards facing an adventurer. At the end of a fall, a creature takes 1d6 bludgeoning
damage for every 10 feet it fell, to a maximum of 20d6. The creature lands prone, unless it avoids taking
damage from the fall. Suffocating A creature can hold its breath for a number of minut

## 2. The 25 questions and their gold passages

The first 8 questions were picked because retrieval is expected to struggle with them (keyword traps, exceptions, negation).
The other 17 are ordinary rules questions, some of them "neutral" ones where no problem is expected, so the results are not
only about hand-picked failures.

For each question the **gold passage** is the part of the SRD that answers it, chosen by hand. A short quote from that passage is noted next to the question, and a chunk counts as
a *gold chunk* if it contains that quote. The cell below prints the passage each quote matches, so every choice can be
checked by eye.

In [3]:
# 25 test questions: (type, question, short quote from the hand-picked passage that answers it)
TESTS = [
 # --- first 8 questions ---
 ("Keyword trap",          "What breaks my focus on a spell?",                          r"incapacitated or killed"),
 ("Negation / constraint", "Do I need to see a target to attack it?",                   r"target that you can.t see"),
 ("Exception / corner",    "Can I punish an unseen enemy that runs away from me?",      r"hostile creature that you can see moves out of your reach"),
 ("Negation / constraint", "Can I cast a spell while a silence effect is on me?",       r"can.t cast a spell with a verbal component"),
 ("Exception / corner",    "Can I shove a creature much bigger than me?",               r"push it away from you"),
 ("Keyword trap (synonym)","How many magic items can one creature bond with?",          r"no more than three magic items"),
 ("Counter-example",       "Can I rest twice in a single day to recover everything?",   r"more than one long rest in a 24"),
 ("Counter-example",       "Can a creature that is held by an enemy still walk away?",  r"grappled creature.s speed becomes 0"),
 # --- 17 new ---
 ("Exception / corner",    "Does teleporting away from an enemy provoke an attack?",    r"don.t provoke an opportunity attack when you teleport"),
 ("Negation / constraint", "Can a surprised character act in the first round?",         r"can.t move or take an action"),
 ("Neutral rule",          "What happens if a character plunges off a cliff?",          r"damage for every 10 feet it fell"),
 ("Neutral rule",          "When does a character die outright instead of falling unconscious?", r"Massive damage can kill you instantly"),
 ("Exception / corner",    "Can I cast a second leveled spell after a bonus action spell?", r"except for a cantrip with a casting time of 1 action"),
 ("Negation / constraint", "Can I concentrate on two spells at once?",                  r"on two spells at once"),
 ("Neutral rule",          "Can I recover from exhaustion by resting?",                 r"exhaustion level by 1"),
 ("Neutral rule",          "How long can a creature hold its breath?",                  r"hold its breath for a number of minutes"),
 ("Exception / corner",    "Can two clerics cast the same blessing on one ally to stack it?", r"most potent effect"),
 ("Exception / corner",    "Can I disengage and still be hit as I leave?",              r"avoid provoking an opportunity attack"),
 ("Negation / constraint", "Can I take two reactions before my next turn?",             r"can.t take another one until the start of your next turn"),
 ("Neutral rule",          "Is a paralyzed creature able to act?",                      r"paralyzed creature is incapacitated"),
 ("Negation / constraint", "Can I double my proficiency bonus twice on the same roll?", r"can.t be added to a single die roll or other number more than once"),
 ("Neutral rule",          "Can advantage and disadvantage cancel each other out?",     r"neither advantage nor disadvantage"),
 ("Exception / corner",    "Can the Dash action be used as a bonus action?",            r"only to take the Dash, Disengage, or Hide"),
 ("Neutral rule",          "Can a prone creature stand up for free?",                   r"Standing up takes more effort"),
 ("Exception / corner",    "Can a monster use a legendary action in the middle of its own turn?", r"called legendary actions"),
]

In [4]:
def gold_chunks(gold):
    g = re.compile(gold, re.I)
    return [c["id"] for c in chunks if g.search(c["text"])]

def gold_rank(order_ids, gold_ids):
    for r, cid in enumerate(order_ids, 1):
        if cid in gold_ids:
            return r
    return None

GOLD = {q: gold_chunks(g) for _, q, g in TESTS}
for n, (cat, q, g) in enumerate(TESTS, 1):
    first = GOLD[q][0]
    m = re.search(g, chunks[first]["text"], re.I)
    ctx = chunks[first]["text"][max(0, m.start() - 90): m.end() + 90].replace("\n", " ")
    print(f"{n:>2}. [{cat}] {q}\n    gold chunk(s) {GOLD[q]}  page {chunks[first]['page']}: ...{ctx}...")
assert all(0 < len(v) <= 4 for v in GOLD.values()), "a gold quote matches nothing, or far too many chunks"

 1. [Keyword trap] What breaks my focus on a spell?
    gold chunk(s) [372]  page 102: ...nd a dragon’s breath, you make a separate saving throw for each source of damage. • Being incapacitated or killed. You lose concentration on a spell if you are incapacitated or if you die. The GM might a...
 2. [Negation / constraint] Do I need to see a target to attack it?
    gold chunk(s) [337]  page 94: ...tice by hiding, casting the invisibility spell, or lurking in darkness. When you attack a target that you can’t see, you have disadvantage on the attack roll. This is true whether you’re guessing the targe...
 3. [Exception / corner] Can I punish an unseen enemy that runs away from me?
    gold chunk(s) [340]  page 95: ... Such a strike is called an opportunity attack. You can make an opportunity attack when a hostile creature that you can see moves out of your reach. To make the opportunity attack, you use your reaction to make one melee attack against t...
 4. [Negation / constraint] Can I

## 3. The two pipelines: bi-encoder retrieval and cross-encoder re-ranking

For each embedder:

* **Stage 1 (bi-encoder).** Every chunk is embedded once and L2-normalised, so the inner product equals cosine similarity. The vectors go into an exact FAISS inner-product index. A question is embedded the same way and its 10 nearest chunks are returned. This is the whole of **Pipeline 1**'s retrieval, and its first 3 chunks are what the LLM sees.
* **Stage 2 (cross-encoder).** The cross-encoder reads the question and each of those same 10 chunks together as one input and gives a relevance score. Sorting by that score and keeping the first 3 gives **Pipeline 2**'s context. The code asserts that the candidate set is unchanged.

Each model expects its own text prefixes: BGE adds an instruction to *queries* only, EmbeddingGemma 2 has a query prefix and a document prefix.
EmbeddingGemma 2 is run in float32 because its model card warns that float16 can give NaN.

Stage timings are measured here too (GPU warm-up first, CUDA synchronised before each timer stop).

In [5]:
EMBEDDERS = {
    "bge-small": dict(name="BAAI/bge-small-en-v1.5",
                      q_prefix="Represent this sentence for searching relevant passages: ", d_prefix="", kwargs={}),
    "embeddinggemma-2": dict(name="google/embeddinggemma-2",
                      q_prefix="task: search result | query: ", d_prefix="title: none | text: ",
                      kwargs={"torch_dtype": torch.float32}),
}
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)     # stage 2, shared by both embedders

def timeit(fn, reps=10):
    """mean milliseconds per call after one warm-up call"""
    fn(); torch.cuda.synchronize(); ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    return 1000 * float(np.mean(ts))

retrieval, emb_info = {}, {}
for key, cfg in EMBEDDERS.items():
    model = SentenceTransformer(cfg["name"], device=DEVICE, model_kwargs=cfg["kwargs"])
    t0 = time.perf_counter()
    E = model.encode([cfg["d_prefix"] + c["text"] for c in chunks], normalize_embeddings=True, batch_size=32)
    torch.cuda.synchronize(); t_index = time.perf_counter() - t0
    assert np.isfinite(E).all(), f"{key}: NaN or inf in the embeddings"
    index = faiss.IndexFlatIP(E.shape[1]); index.add(E.astype("float32"))

    def stage1(q, k=10):
        qv = model.encode([cfg["q_prefix"] + q], normalize_embeddings=True).astype("float32")
        _, ids = index.search(qv, k)
        return [int(i) for i in ids[0]]

    retrieval[key] = {}
    for cat, q, g in TESTS:
        base = stage1(q)                                                   # Pipeline 1 uses base[:3]
        sc = ce.predict([(q, chunks[i]["text"]) for i in base], batch_size=10)
        order = [base[j] for j in np.argsort(-sc)]                         # Pipeline 2 uses order[:3]
        assert set(order) == set(base)
        retrieval[key][q] = dict(category=cat, base_ids=base, rerank_ids=order, gold_ids=GOLD[q],
                                 gold_rank_base=gold_rank(base, GOLD[q]), gold_rank_rerank=gold_rank(order, GOLD[q]),
                                 ce={base[j]: float(sc[j]) for j in range(10)})
    qs_all = [q for _, q, _ in TESTS]
    fixed = {q: stage1(q) for q in qs_all}                                 # fixed candidates, so stage 2 is timed on its own
    emb_info[key] = dict(dim=int(E.shape[1]), index_seconds=round(t_index, 1),
        stage1_ms=round(timeit(lambda: [stage1(q) for q in qs_all]) / len(qs_all), 1),
        stage2_ms=round(timeit(lambda: [ce.predict([(q, chunks[i]["text"]) for i in fixed[q]], batch_size=10) for q in qs_all]) / len(qs_all), 1))
    print(key, emb_info[key])
    del model, index, E; gc.collect(); torch.cuda.empty_cache()             # free the GPU before the next model

json.dump(dict(retrieval=retrieval, emb_info=emb_info, chunks=chunks), open(f"{OUT}/retrieval.json", "w"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

bge-small {'dim': 384, 'index_seconds': 4.9, 'stage1_ms': 10.5, 'stage2_ms': 17.3}


Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

embeddinggemma-2 {'dim': 768, 'index_seconds': 26.2, 'stage1_ms': 56.9, 'stage2_ms': 17.4}


## 4. The retrieved chunks, saved in readable form

`retrieved_chunks.md` lists, for every question and both embedders, the ten bi-encoder results, their order after the
cross-encoder, scores, page numbers and the chunk text. `retrieval.json` holds the same ids plus all chunks.

In [6]:
def ce_score(r, cid):
    return r["ce"].get(cid, r["ce"].get(str(cid)))

lines = ["# Retrieved chunks for the 25 test questions", "",
         "For each question and embedder: the ten chunks returned by the bi-encoder, their order after the cross-encoder, and the chunk text.",
         "Pipeline 1 gives the LLM the first 3 of the bi-encoder order, Pipeline 2 the first 3 of the cross-encoder order.",
         "`GOLD` marks a chunk that contains the passage answering the question. The cross-encoder score is a raw logit (higher is more relevant, it can be negative)."]
for n, (cat, q, g) in enumerate(TESTS, 1):
    lines += ["", f"## Q{n}. {q}"]
    for e in EMBEDDERS:
        r = retrieval[e][q]; gold = set(r["gold_ids"]); bi, rr = r["base_ids"], r["rerank_ids"]
        lines += ["", f"### {e} | gold chunk(s): {', '.join('c%d' % i for i in sorted(gold))} | gold rank {r['gold_rank_base']} -> {r['gold_rank_rerank']}", "",
                  "| Re-ranked # | Bi-encoder # | Chunk | Page | Cross-encoder score | In baseline top-3 | In re-ranked top-3 | Gold |",
                  "|---|---|---|---|---|---|---|---|"]
        for k, cid in enumerate(rr, 1):
            b = bi.index(cid) + 1
            lines.append(f"| {k} | {b} | c{cid} | {chunks[cid]['page']} | {ce_score(r, cid):.2f} | {'yes' if b <= 3 else 'no'} | {'yes' if k <= 3 else 'no'} | {'GOLD' if cid in gold else ''} |")
        lines += ["", "<details><summary>Chunk texts</summary>", ""]
        for cid in sorted(bi):
            lines += [f"**c{cid} (page {chunks[cid]['page']})**  ", " ".join(chunks[cid]["text"].split()), ""]
        lines.append("</details>")
open(f"{OUT}/retrieved_chunks.md", "w", encoding="utf-8").write("\n".join(lines))
print("saved", f"{OUT}/retrieved_chunks.md", f"({os.path.getsize(OUT + '/retrieved_chunks.md') / 1e3:.0f} KB)")

saved /home2/akshat.pandey/rag/outputs_final/retrieved_chunks.md (625 KB)


## 5. Retrieval metrics

* **recall@3**: questions whose gold chunk is in the 3 chunks the LLM would see.
* **recall@10**: questions whose gold chunk is anywhere among the 10 candidates. Re-ranking cannot change this number, and it is the ceiling for recall@3 after re-ranking.
* **rescued**: gold chunk moved from outside the top 3 into it. **lost**: moved from inside to outside.
* **MRR**: mean of 1 / rank of the gold chunk (0 when it is not in the top 10).

In [7]:
def mrr(ranks): return float(np.mean([1 / r if r else 0 for r in ranks]))
rows = []
for key in EMBEDDERS:
    R = list(retrieval[key].values())
    rb = [x["gold_rank_base"] for x in R]; rr = [x["gold_rank_rerank"] for x in R]
    inb = [r is not None and r <= 3 for r in rb]; inr = [r is not None and r <= 3 for r in rr]
    both = [(b, a) for b, a in zip(rb, rr) if b and a]
    rows.append({"embedder": key, "questions": len(R),
                 "recall@3 baseline": sum(inb), "recall@3 re-ranked": sum(inr), "recall@10": sum(r is not None for r in rb),
                 "rescued (out of top 3 -> in)": sum((not b) and a for b, a in zip(inb, inr)),
                 "lost (in -> out)": sum(b and (not a) for b, a in zip(inb, inr)),
                 "MRR baseline": round(mrr(rb), 3), "MRR re-ranked": round(mrr(rr), 3),
                 "rank improved": sum(a < b for b, a in both), "rank worsened": sum(a > b for b, a in both),
                 "rank unchanged": sum(a == b for b, a in both)})
metrics = pd.DataFrame(rows).set_index("embedder").T
metrics.to_csv(f"{OUT}/metrics.csv")
metrics

embedder,bge-small,embeddinggemma-2
questions,25.000,25.000
recall@3 baseline,18.000,16.000
recall@3 re-ranked,21.000,21.000
recall@10,22.000,23.000
rescued (out of top 3 -> in),3.000,6.000
lost (in -> out),0.000,1.000
MRR baseline,0.613,0.645
MRR re-ranked,0.711,0.685
rank improved,8.000,8.000
rank worsened,3.000,3.000


## 6. Rank shifts per question, by question type, and the cost of running the cross-encoder on everything

The first table shows the rank of the gold chunk before and after re-ranking (`None` = not among the ten candidates).
The last cell times the cross-encoder over **every** chunk for one question. That is the cost of dropping the bi-encoder
stage, and it feeds the scaling discussion in the report.

In [8]:
pq = pd.DataFrame([{"#": n, "type": cat, "question": q,
                    **{f"{k}: baseline -> re-ranked": f"{retrieval[k][q]['gold_rank_base']} -> {retrieval[k][q]['gold_rank_rerank']}" for k in EMBEDDERS}}
                   for n, (cat, q, g) in enumerate(TESTS, 1)])
pd.set_option("display.width", 250, "display.max_colwidth", 80, "display.max_rows", 100)
pq.to_csv(f"{OUT}/per_question_ranks.csv", index=False)
pq

,#,type,question,bge-small: baseline -> re-ranked,embeddinggemma-2: baseline -> re-ranked
0,1,Keyword trap,What breaks my focus on a spell?,5 -> 2,6 -> 2
1,2,Negation / constraint,Do I need to see a target to attack it?,8 -> 3,5 -> 2
2,3,Exception / corner,Can I punish an unseen enemy that runs away from me?,4 -> 3,5 -> 5
3,4,Negation / constraint,Can I cast a spell while a silence effect is on me?,3 -> 1,6 -> 1
4,5,Exception / corner,Can I shove a creature much bigger than me?,3 -> 1,None -> None
5,6,Keyword trap (synonym),How many magic items can one creature bond with?,3 -> 1,1 -> 1
6,7,Counter-example,Can I rest twice in a single day to recover everything?,None -> None,5 -> 2
7,8,Counter-example,Can a creature that is held by an enemy still walk away?,None -> None,None -> None
8,9,Exception / corner,Does teleporting away from an enemy provoke an attack?,1 -> 1,2 -> 1
9,10,Negation / constraint,Can a surprised character act in the first round?,1 -> 1,1 -> 1


In [9]:
by = []
for key in EMBEDDERS:
    for cat in sorted({c for c, _, _ in TESTS}):
        R = [x for x in retrieval[key].values() if x["category"] == cat]
        by.append({"embedder": key, "type": cat, "questions": len(R),
                   "recall@3 baseline": sum(x["gold_rank_base"] is not None and x["gold_rank_base"] <= 3 for x in R),
                   "recall@3 re-ranked": sum(x["gold_rank_rerank"] is not None and x["gold_rank_rerank"] <= 3 for x in R)})
by = pd.DataFrame(by); by.to_csv(f"{OUT}/by_question_type.csv", index=False); by

,embedder,type,questions,recall@3 baseline,recall@3 re-ranked
0,bge-small,Counter-example,2,0,0
1,bge-small,Exception / corner,8,6,7
2,bge-small,Keyword trap,1,0,1
3,bge-small,Keyword trap (synonym),1,1,1
4,bge-small,Negation / constraint,6,5,6
5,bge-small,Neutral rule,7,6,6
6,embeddinggemma-2,Counter-example,2,0,1
7,embeddinggemma-2,Exception / corner,8,5,6
8,embeddinggemma-2,Keyword trap,1,0,1
9,embeddinggemma-2,Keyword trap (synonym),1,1,1


In [10]:
full = [(TESTS[0][1], c["text"]) for c in chunks]
t_all_ms = timeit(lambda: ce.predict(full, batch_size=128), reps=3)
per_chunk_ms = t_all_ms / len(chunks)
print(f"cross-encoder over all {len(chunks)} chunks for one question: {t_all_ms:.0f} ms  ({per_chunk_ms:.2f} ms per chunk)")
scaling = pd.DataFrame([{"chunks in corpus": n, "cross-encoder over everything (s per question)": round(per_chunk_ms * n / 1000, 1)}
                        for n in (len(chunks), 100_000, 1_000_000, 10_000_000)])
scaling.to_csv(f"{OUT}/cross_encoder_scaling.csv", index=False)
print("re-ranking only the top 10 stays at about", round(sum(e["stage2_ms"] for e in emb_info.values()) / len(emb_info), 1), "ms regardless of corpus size")
scaling

cross-encoder over all 1504 chunks for one question: 1884 ms  (1.25 ms per chunk)
re-ranking only the top 10 stays at about 17.4 ms regardless of corpus size


,chunks in corpus,cross-encoder over everything (s per question)
0,1504,1.9
1,100000,125.3
2,1000000,1252.9
3,10000000,12528.9


## 7. The LLM: Qwen3.5-9B split across two GPUs

The model is about 18 GB in fp16, more than one 11 GB card, so `device_map="auto"` places its layers on both GPUs.
**Thinking is off** (greedy decoding). An earlier attempt with thinking on generated about 12 tokens per second and 15 of 16
tries used the whole 2,500-token allowance without finishing, so it was not evaluated (`outputs/thinking_attempt_partial.json` keeps the evidence).

The prompt is identical for both pipelines. It tells the model to answer only from the passages and to say
"The provided context does not answer this." when they do not contain the answer.

In [11]:
PROMPT = """You are a rules assistant for Dungeons & Dragons 5th edition.
Answer the question using ONLY the context passages below. If the passages do not contain the answer, say
"The provided context does not answer this." Be concise (at most 4 sentences) and state any exception or restriction that applies.

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(query, ids):
    ctx = "\n\n".join(f"[Passage {n+1}, page {chunks[i]['page']}] {chunks[i]['text']}" for n, i in enumerate(ids))
    return PROMPT.format(context=ctx, question=query)

tok = AutoTokenizer.from_pretrained(LLM_NAME)
t0 = time.perf_counter()
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="auto", max_memory={0: "10GiB", 1: "10GiB"})
llm.eval()
print(f"loaded in {time.perf_counter()-t0:.0f}s | class {type(llm).__name__}")
for i in range(torch.cuda.device_count()):
    print(f"gpu{i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB allocated of {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}")

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

loaded in 155s | class Qwen3_5ForCausalLM
gpu0: 8.6 GB allocated of 11.3
gpu1: 9.4 GB allocated of 11.3


In [12]:
gen_cache = {}
def generate(prompt):
    """Greedy answer with thinking off. Identical prompts are generated once and reused."""
    if prompt in gen_cache:
        return dict(gen_cache[prompt], cached=True)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
    inputs = tok([text], return_tensors="pt").to("cuda:0")
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    t = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(**inputs, do_sample=False, max_new_tokens=250)
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    secs = time.perf_counter() - t
    new = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(new, skip_special_tokens=True)
    answer = raw.split("</think>", 1)[1] if "</think>" in raw else raw
    res = dict(answer=answer.strip(), new_tokens=int(len(new)), seconds=secs, prompt_tokens=int(inputs["input_ids"].shape[1]))
    gen_cache[prompt] = res
    return dict(res, cached=False)

# sanity check: the fp16 two-GPU model must produce sensible text before the real run
chk = generate("Reply with just the number: what is 12 plus 30?")
print(repr(chk["answer"]), f"{chk['seconds']:.1f}s")
assert "42" in chk["answer"], "model output looks broken (fp16 problem?)"

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


'42' 3.6s


## 8. Run both pipelines on every question, for both embedders

For each embedder and question: **Answer 1** comes from the baseline top 3 and **Answer 2** from the re-ranked top 3, with the identical prompt.
That is 25 questions x 2 embedders x 2 pipelines = 100 answers.

In [13]:
QUERIES = [q for _, q, _ in TESTS]
EMBS = list(EMBEDDERS)
results, started = [], time.perf_counter()
for emb in EMBS:
    for q in QUERIES:
        r = retrieval[emb][q]
        for pipeline, ids in (("baseline", r["base_ids"][:3]), ("rerank", r["rerank_ids"][:3])):
            g = generate(build_prompt(q, ids))
            results.append(dict(embedder=emb, category=r["category"], query=q, pipeline=pipeline, context_ids=ids,
                                gold_in_context=any(c in ids for c in r["gold_ids"]), **g))
    print(f"done: {emb} | elapsed {(time.perf_counter()-started)/60:.1f} min", flush=True)
json.dump(dict(results=results, llm=LLM_NAME), open(f"{OUT}/generations_qwen35_9b.json", "w"), indent=1)
fresh = [x for x in results if not x["cached"]]
print(f"{len(results)} answers ({len(fresh)} generated, {len(results) - len(fresh)} reused because the context was identical); "
      f"mean {np.mean([x['seconds'] for x in fresh]):.1f} s and {np.mean([x['new_tokens'] for x in fresh]):.0f} tokens per generated answer")

done: bge-small | elapsed 6.1 min


done: embeddinggemma-2 | elapsed 10.9 min


100 answers (92 generated, 8 reused because the context was identical); mean 7.1 s and 72 tokens per generated answer


## 9. The answers

Every answer, grouped by embedder and question. `ctx_has_gold` says whether the gold chunk was among the 3 chunks the model was shown.

In [14]:
for emb in EMBS:
    print("#" * 100, f"\nEMBEDDER: {emb}")
    for n, q in enumerate(QUERIES, 1):
        r = retrieval[emb][q]
        print("-" * 100, f"\nQ{n}. {q}   (gold rank {r['gold_rank_base']} -> {r['gold_rank_rerank']})")
        for p in ("baseline", "rerank"):
            x = next(y for y in results if y["embedder"] == emb and y["query"] == q and y["pipeline"] == p)
            print(f"  [{p:8s}] ctx_has_gold={x['gold_in_context']!s:5s}", textwrap.fill(x["answer"], 118, subsequent_indent="      ", initial_indent=""))

#################################################################################################### 
EMBEDDER: bge-small
---------------------------------------------------------------------------------------------------- 
Q1. What breaks my focus on a spell?   (gold rank 5 -> 2)
  [baseline] ctx_has_gold=False Taking damage while concentrating on a spell breaks your focus unless you succeed on a Constitution saving throw.
      Additionally, casting another spell that requires concentration immediately ends your current concentration. You
      can also voluntarily end concentration at any time without using an action. Normal activities, such as moving or
      attacking, do not interfere with concentration.
  [rerank  ] ctx_has_gold=True  You lose concentration on a spell if you cast another spell that requires concentration, become incapacitated, or die.
      Additionally, taking damage forces you to make a Constitution saving throw (DC 10 or half the damage, whichever
      is hi

## 10. Grading the answers against gold key facts

The gold key facts below are my own summary of what the SRD says for each question. The verdicts were assigned **by hand** after
reading every answer, so they are a judgement, not an automatic score:

* **C** correct: the core rule is right and nothing in it is wrong.
* **P** partly correct: the core is right but a key clause is missing, the opening "yes/no" is confusing, or a detail is invented or wrong.
* **W** wrong rule: the answer states a rule that is false.
* **A** abstains: the model says "The provided context does not answer this."

Each verdict is stored together with a fingerprint of the answer text it was given for. The cell prints how many of the answers
generated in this run are identical to the graded ones; any that differ would need to be re-graded.

In [15]:
GOLD_FACTS = [
 "Another concentration spell; damage (Con save, DC 10 or half the damage); incapacitated or killed; GM-called DC 10 save for environmental effects; you can end it at will; normal activity is fine.",
 "No. You can attack a target you cannot see, but with disadvantage; if it is not where you aimed, the attack misses.",
 "No. An opportunity attack needs a hostile creature that you can see moving out of your reach.",
 "Not a spell with a verbal component. Spells without V can still be cast.",
 "No. The target of a shove must be no more than one size larger than you.",
 "Three at a time; a fourth attunement fails until you end one; no two copies of the same item.",
 "No. A character cannot benefit from more than one long rest in a 24-hour period.",
 "No. A grappled creature's speed becomes 0 (it can try to escape the grapple instead).",
 "No. Teleporting does not provoke an opportunity attack.",
 "No. A surprised creature cannot move or take an action on its first turn and cannot take a reaction until that turn ends.",
 "1d6 bludgeoning damage per 10 feet fallen, maximum 20d6, and you land prone.",
 "When damage takes you to 0 HP and the remaining damage is at least your hit point maximum (massive damage).",
 "No. After casting a spell with a bonus action you can only cast a cantrip with a casting time of 1 action that turn.",
 "No. You cannot concentrate on two spells at once.",
 "Yes. Finishing a long rest reduces exhaustion by 1 (with some food and drink).",
 "1 + Constitution modifier minutes (minimum 30 seconds).",
 "No. The same spell cast several times does not combine; the most potent effect applies.",
 "No. Taking the Disengage action means your movement does not provoke opportunity attacks.",
 "No. You cannot take another reaction until the start of your next turn.",
 "No. Paralyzed means incapacitated and unable to move or speak.",
 "No. Your proficiency bonus cannot be added to a single roll more than once.",
 "Yes. If you have both, they cancel and you roll one d20, however many sources of each there are.",
 "Only through a specific feature, such as the rogue's Cunning Action; otherwise Dash is an action.",
 "No. Standing up costs movement equal to half your speed.",
 "No. Legendary actions are used only at the end of another creature's turn, one at a time.",
]
# verdicts per question 1..25, in the order of the questions above
VERDICTS = {
 "bge-small|baseline":        "P W A C C C C A C C A C C C C C C P C C C C P C C",
 "bge-small|rerank":          "C C P C C C A P C P A C C C C C C P C C C C C C C",
 "embeddinggemma-2|baseline": "P P A P A C P A C C C C C C A C C P C C C C P C C",
 "embeddinggemma-2|rerank":   "C C A C A C C P C P A C C C P C C P C C P C C C C",
}
GRADED_SHA = {
    "bge-small|baseline|0": "a25b4c3c", "bge-small|baseline|1": "1cfbd573", "bge-small|baseline|2": "ba40af2f", "bge-small|baseline|3": "aafdd385", "bge-small|baseline|4": "c9aa60ac", "bge-small|baseline|5": "502adb3a", "bge-small|baseline|6": "450b0f70", "bge-small|baseline|7": "ba40af2f", "bge-small|baseline|8": "25a53200", "bge-small|baseline|9": "b781c037", "bge-small|baseline|10": "ba40af2f", "bge-small|baseline|11": "75f35fc2", "bge-small|baseline|12": "ab1a245a", "bge-small|baseline|13": "d59e5bd0", "bge-small|baseline|14": "10a41f56", "bge-small|baseline|15": "16869d3f", "bge-small|baseline|16": "42095e16", "bge-small|baseline|17": "6c821009", "bge-small|baseline|18": "74a5ae09", "bge-small|baseline|19": "99baa4a6", "bge-small|baseline|20": "8abaf17d", "bge-small|baseline|21": "8c0c992f", "bge-small|baseline|22": "e3e75f7e", "bge-small|baseline|23": "429f733f", "bge-small|baseline|24": "ec6a7171",
    "bge-small|rerank|0": "f83e29f8", "bge-small|rerank|1": "14c48e04", "bge-small|rerank|2": "f5d6c058", "bge-small|rerank|3": "43eae48e", "bge-small|rerank|4": "ce736232", "bge-small|rerank|5": "7c66934d", "bge-small|rerank|6": "ba40af2f", "bge-small|rerank|7": "33a9acb4", "bge-small|rerank|8": "6f5d08aa", "bge-small|rerank|9": "13bce116", "bge-small|rerank|10": "5ef836b9", "bge-small|rerank|11": "74bbc3e7", "bge-small|rerank|12": "4e568d3b", "bge-small|rerank|13": "77a4c90e", "bge-small|rerank|14": "820a2f27", "bge-small|rerank|15": "5a401146", "bge-small|rerank|16": "2333886e", "bge-small|rerank|17": "f4a47141", "bge-small|rerank|18": "90c6072c", "bge-small|rerank|19": "21cc0bdb", "bge-small|rerank|20": "6a89bcc6", "bge-small|rerank|21": "bc2fcc15", "bge-small|rerank|22": "bb31e62d", "bge-small|rerank|23": "476b9516", "bge-small|rerank|24": "317c2259",
    "embeddinggemma-2|baseline|0": "633193a7", "embeddinggemma-2|baseline|1": "cf3cacb2", "embeddinggemma-2|baseline|2": "ba40af2f", "embeddinggemma-2|baseline|3": "7d2df8a3", "embeddinggemma-2|baseline|4": "ba40af2f", "embeddinggemma-2|baseline|5": "c9263caf", "embeddinggemma-2|baseline|6": "8616454b", "embeddinggemma-2|baseline|7": "ba40af2f", "embeddinggemma-2|baseline|8": "de81976a", "embeddinggemma-2|baseline|9": "b781c037", "embeddinggemma-2|baseline|10": "a0944f8a", "embeddinggemma-2|baseline|11": "03bb0370", "embeddinggemma-2|baseline|12": "8cbcc409", "embeddinggemma-2|baseline|13": "b0e9dce5", "embeddinggemma-2|baseline|14": "fdd909f4", "embeddinggemma-2|baseline|15": "eea458af", "embeddinggemma-2|baseline|16": "c944a9aa", "embeddinggemma-2|baseline|17": "cf021f56", "embeddinggemma-2|baseline|18": "2d3ddc00", "embeddinggemma-2|baseline|19": "7fb89b71", "embeddinggemma-2|baseline|20": "15a0267b", "embeddinggemma-2|baseline|21": "d990d1ab", "embeddinggemma-2|baseline|22": "2b324388", "embeddinggemma-2|baseline|23": "4acee06e", "embeddinggemma-2|baseline|24": "ec6a7171",
    "embeddinggemma-2|rerank|0": "f83e29f8", "embeddinggemma-2|rerank|1": "506243fb", "embeddinggemma-2|rerank|2": "ba40af2f", "embeddinggemma-2|rerank|3": "30615cd8", "embeddinggemma-2|rerank|4": "ba40af2f", "embeddinggemma-2|rerank|5": "bd5fc964", "embeddinggemma-2|rerank|6": "0482c9b2", "embeddinggemma-2|rerank|7": "c8000cd1", "embeddinggemma-2|rerank|8": "6f5d08aa", "embeddinggemma-2|rerank|9": "ceb89110", "embeddinggemma-2|rerank|10": "ba40af2f", "embeddinggemma-2|rerank|11": "edfda00d", "embeddinggemma-2|rerank|12": "490e2110", "embeddinggemma-2|rerank|13": "b0e9dce5", "embeddinggemma-2|rerank|14": "e1a0581d", "embeddinggemma-2|rerank|15": "5a401146", "embeddinggemma-2|rerank|16": "7d1f7da3", "embeddinggemma-2|rerank|17": "03c80a72", "embeddinggemma-2|rerank|18": "90c6072c", "embeddinggemma-2|rerank|19": "99baa4a6", "embeddinggemma-2|rerank|20": "d2de58a7", "embeddinggemma-2|rerank|21": "d990d1ab", "embeddinggemma-2|rerank|22": "8e386ec7", "embeddinggemma-2|rerank|23": "7692dfd4", "embeddinggemma-2|rerank|24": "317c2259",
}

def sha(text): return hashlib.sha1(text.strip().encode("utf-8")).hexdigest()[:8]

rows = []
for emb in EMBS:
    for p in ("baseline", "rerank"):
        for i, q in enumerate(QUERIES):
            x = next(y for y in results if y["embedder"] == emb and y["query"] == q and y["pipeline"] == p)
            rows.append(dict(embedder=emb, pipeline=p, q=i + 1, question=q, verdict=VERDICTS[f"{emb}|{p}"].split()[i],
                             same_answer_as_graded=GRADED_SHA.get(f"{emb}|{p}|{i}") == sha(x["answer"]),
                             gold_in_context=x["gold_in_context"], answer=x["answer"]))
G = pd.DataFrame(rows)
G.to_csv(f"{OUT}/answers_graded.csv", index=False)
print(f"answers identical to the graded ones: {int(G.same_answer_as_graded.sum())} of {len(G)}")
if not G.same_answer_as_graded.all():
    print("NOT identical (these need re-grading):\n", G[~G.same_answer_as_graded][["embedder", "pipeline", "q"]].to_string(index=False))

def verdict_table(lo, hi):
    sub = G[(G.q >= lo) & (G.q <= hi)]
    t = pd.crosstab([sub.embedder, sub.pipeline], sub.verdict).reindex(columns=list("CPWA"), fill_value=0)
    t.columns = ["Correct", "Partial", "Wrong rule", "Abstain"]
    return t

print("\nAll 25 questions"); display(verdict_table(1, 25))
print("First 8 questions (picked because retrieval struggles)"); display(verdict_table(1, 8))
print("Other 17 questions (ordinary)"); display(verdict_table(9, 25))

answers identical to the graded ones: 100 of 100

All 25 questions


Correct  Partial  Wrong rule  Abstain
embedder         pipeline                                       
bge-small        baseline       18        3           1        3
                 rerank         19        4           0        2
embeddinggemma-2 baseline       15        6           0        4
                 rerank         17        5           0        3

First 8 questions (picked because retrieval struggles)


Correct  Partial  Wrong rule  Abstain
embedder         pipeline                                       
bge-small        baseline        4        1           1        2
                 rerank          5        2           0        1
embeddinggemma-2 baseline        1        4           0        3
                 rerank          5        1           0        2

Other 17 questions (ordinary)


Correct  Partial  Wrong rule  Abstain
embedder         pipeline                                       
bge-small        baseline       14        2           0        1
                 rerank         14        2           0        1
embeddinggemma-2 baseline       14        2           0        1
                 rerank         12        4           0        1

In [16]:
# per question: gold key facts and the four verdicts (B = baseline top 3, R = re-ranked top 3)
pv = G.pivot_table(index="q", columns=["embedder", "pipeline"], values="verdict", aggfunc="first")
pv.columns = [f"{e} {'B' if p == 'baseline' else 'R'}" for e, p in pv.columns]
pv.insert(0, "gold key facts", GOLD_FACTS)
pv.insert(0, "question", QUERIES)
pv.to_csv(f"{OUT}/verdicts_per_question.csv")
pd.set_option("display.max_colwidth", 90)
pv

,question,gold key facts,bge-small B,bge-small R,embeddinggemma-2 B,embeddinggemma-2 R
q,,,,,,
1,What breaks my focus on a spell?,"Another concentration spell; damage (Con save, DC 10 or half the damage); incapacitate...",P,C,P,C
2,Do I need to see a target to attack it?,"No. You can attack a target you cannot see, but with disadvantage; if it is not where ...",W,C,P,C
3,Can I punish an unseen enemy that runs away from me?,No. An opportunity attack needs a hostile creature that you can see moving out of your...,A,P,A,A
4,Can I cast a spell while a silence effect is on me?,Not a spell with a verbal component. Spells without V can still be cast.,C,C,P,C
5,Can I shove a creature much bigger than me?,No. The target of a shove must be no more than one size larger than you.,C,C,A,A
6,How many magic items can one creature bond with?,Three at a time; a fourth attunement fails until you end one; no two copies of the sam...,C,C,C,C
7,Can I rest twice in a single day to recover everything?,No. A character cannot benefit from more than one long rest in a 24-hour period.,C,A,P,C
8,Can a creature that is held by an enemy still walk away?,No. A grappled creature's speed becomes 0 (it can try to escape the grapple instead).,A,P,A,P
9,Does teleporting away from an enemy provoke an attack?,No. Teleporting does not provoke an opportunity attack.,C,C,C,C


## 11. Latency: baseline pipeline vs re-ranking pipeline

* **Stage 1** is embedding the question and the FAISS search (section 3). **Stage 2** is the cross-encoder on the 10 candidates (section 3), which only the re-ranking pipeline runs.
* **LLM** is the mean time Qwen3.5-9B took per answer for that pipeline (section 8). Where the baseline and re-ranked contexts were identical the answer was reused and carries the time of the one generation.
* **Total** = stage 1 + stage 2 + LLM. Stage times are per question; times in milliseconds.

In [17]:
gen = pd.DataFrame(results)
lat_rows = []
for emb in EMBS:
    for p in ("baseline", "rerank"):
        s1 = emb_info[emb]["stage1_ms"]
        s2 = emb_info[emb]["stage2_ms"] if p == "rerank" else 0.0
        llm_ms = 1000 * gen[(gen.embedder == emb) & (gen.pipeline == p)].seconds.mean()
        lat_rows.append({"embedder": emb, "pipeline": "baseline (bi-encoder only)" if p == "baseline" else "re-ranking (bi-encoder + cross-encoder)",
                         "stage 1 (ms)": s1, "stage 2 (ms)": s2, "retrieval total (ms)": round(s1 + s2, 1),
                         "LLM per answer (ms)": round(llm_ms), "end to end (ms)": round(s1 + s2 + llm_ms)})
latency = pd.DataFrame(lat_rows)
latency.to_csv(f"{OUT}/latency.csv", index=False)
for emb in EMBS:
    b, r = latency[latency.embedder == emb].iloc[0], latency[latency.embedder == emb].iloc[1]
    print(f"{emb}: re-ranking adds {r['stage 2 (ms)']:.1f} ms = {100 * r['stage 2 (ms)'] / b['end to end (ms)']:.2f}% of the end-to-end time")
print(f"cross-encoder over ALL {len(chunks)} chunks instead of the top 10: {t_all_ms:.0f} ms per question")
latency

bge-small: re-ranking adds 17.3 ms = 0.25% of the end-to-end time
embeddinggemma-2: re-ranking adds 17.4 ms = 0.26% of the end-to-end time
cross-encoder over ALL 1504 chunks instead of the top 10: 1884 ms per question


,embedder,pipeline,stage 1 (ms),stage 2 (ms),retrieval total (ms),LLM per answer (ms),end to end (ms)
0,bge-small,baseline (bi-encoder only),10.5,0.0,10.5,6813,6823
1,bge-small,re-ranking (bi-encoder + cross-encoder),10.5,17.3,27.8,7724,7752
2,embeddinggemma-2,baseline (bi-encoder only),56.9,0.0,56.9,6676,6733
3,embeddinggemma-2,re-ranking (bi-encoder + cross-encoder),56.9,17.4,74.3,6937,7011


## 12. Failure-mode analysis (stress test)

The assignment asks for at least two failure queries where the baseline stumbles and re-ranking restores relevance, plus the
document-specific failure types below. Everything in this section is computed from the results above.

* **Baseline stumbles:** the gold chunk is outside the baseline top 3 (so the LLM never saw it) and the baseline answer was not graded correct.
* **Re-ranking restores relevance:** after re-ranking the gold chunk is inside the top 3 and the re-ranked answer is graded correct.
* **Buried chunk promoted:** the gold chunk sat at rank 4 to 10 for the bi-encoder and is in the top 3 after the cross-encoder.

The question types map to the assignment's failure types: *Keyword trap* (and its synonym variant) = keyword trap / distractor,
*Exception / corner* = exceptions and corner cases, *Negation / constraint* = negation and constraints; the remaining
questions (*Neutral rule*, *Counter-example*) and every buried-then-promoted case in any group are the "other document-specific failures".

In [18]:
def first_gold_in(ids, gold_ids):
    return next((c for c in ids if c in gold_ids), None)

cases = []
for emb in EMBS:
    for n, (cat, q, g) in enumerate(TESTS, 1):
        r = retrieval[emb][q]; b, a = r["gold_rank_base"], r["gold_rank_rerank"]
        vb = G[(G.embedder == emb) & (G.q == n) & (G.pipeline == "baseline")].verdict.iloc[0]
        vr = G[(G.embedder == emb) & (G.q == n) & (G.pipeline == "rerank")].verdict.iloc[0]
        gold_id = first_gold_in(r["base_ids"], r["gold_ids"])
        cases.append(dict(embedder=emb, q=n, type=cat, question=q, gold_chunk=gold_id,
                          rank_baseline=b, rank_reranked=a,
                          buried_then_promoted=bool(b and 4 <= b <= 10 and a and a <= 3),
                          answer1_verdict=vb, answer2_verdict=vr,
                          restored=bool(b and b > 3 and a and a <= 3 and vb != "C" and vr == "C")))
F = pd.DataFrame(cases)
F.to_csv(f"{OUT}/failure_analysis.csv", index=False)

print("Failure queries: gold chunk buried at rank 4-10 for the bi-encoder and promoted into the top 3 by the cross-encoder\n")
for emb in EMBS:
    sub = F[(F.embedder == emb) & F.buried_then_promoted]
    print(f"[{emb}] {len(sub)} questions")
    for _, x in sub.iterrows():
        print(f"   Q{x.q:<2} [{x.type}] {x.question}\n       Chunk c{int(x.gold_chunk)} moved from Rank #{int(x.rank_baseline)} -> Rank #{int(x.rank_reranked)}"
              f"   | Answer 1: {x.answer1_verdict}   Answer 2: {x.answer2_verdict}   {'RESTORED' if x.restored else ''}")
print("\nverdicts: C correct, P partly correct, W wrong rule, A abstains")
print("\nBaseline stumbles AND re-ranking restores relevance (the assignment's core requirement):")
for emb in EMBS:
    sub = F[(F.embedder == emb) & F.restored]
    print(f"  {emb}: {len(sub)} questions ->", ", ".join(f"Q{q}" for q in sub.q))

Failure queries: gold chunk buried at rank 4-10 for the bi-encoder and promoted into the top 3 by the cross-encoder

[bge-small] 3 questions
   Q1  [Keyword trap] What breaks my focus on a spell?
       Chunk c372 moved from Rank #5 -> Rank #2   | Answer 1: P   Answer 2: C   RESTORED
   Q2  [Negation / constraint] Do I need to see a target to attack it?
       Chunk c337 moved from Rank #8 -> Rank #3   | Answer 1: W   Answer 2: C   RESTORED
   Q3  [Exception / corner] Can I punish an unseen enemy that runs away from me?
       Chunk c340 moved from Rank #4 -> Rank #3   | Answer 1: A   Answer 2: P   
[embeddinggemma-2] 6 questions
   Q1  [Keyword trap] What breaks my focus on a spell?
       Chunk c372 moved from Rank #6 -> Rank #2   | Answer 1: P   Answer 2: C   RESTORED
   Q2  [Negation / constraint] Do I need to see a target to attack it?
       Chunk c337 moved from Rank #5 -> Rank #2   | Answer 1: P   Answer 2: C   RESTORED
   Q4  [Negation / constraint] Can I cast a spell while a 

### 12a. By failure type

For each failure type: the questions, where the gold chunk went, and what happened to the two answers.

In [19]:
GROUPS = {"Keyword trap / distractor": ["Keyword trap", "Keyword trap (synonym)"],
          "Exceptions & corner cases": ["Exception / corner"],
          "Negation & constraints": ["Negation / constraint"],
          "Other document-specific failures": ["Counter-example", "Neutral rule"]}
def rk(x): return "not in top 10" if pd.isna(x) else f"#{int(x)}"
for name, types in GROUPS.items():
    print("=" * 100, f"\n{name}")
    for emb in EMBS:
        sub = F[(F.embedder == emb) & F.type.isin(types)]
        n_stumble = int(((sub.rank_baseline.isna()) | (sub.rank_baseline > 3)).sum())
        print(f"  [{emb}] {len(sub)} questions; gold chunk outside the baseline top 3 in {n_stumble}; "
              f"buried -> promoted in {int(sub.buried_then_promoted.sum())}; baseline answer fully correct in {int((sub.answer1_verdict == 'C').sum())}, "
              f"re-ranked answer fully correct in {int((sub.answer2_verdict == 'C').sum())}")
        for _, x in sub.iterrows():
            print(f"      Q{x.q:<2} {x.question[:62]:<62} gold {rk(x.rank_baseline):>13} -> {rk(x.rank_reranked):<13} Answer 1 {x.answer1_verdict} | Answer 2 {x.answer2_verdict}")

Keyword trap / distractor
  [bge-small] 2 questions; gold chunk outside the baseline top 3 in 1; buried -> promoted in 1; baseline answer fully correct in 1, re-ranked answer fully correct in 2
      Q1  What breaks my focus on a spell?                               gold            #5 -> #2            Answer 1 P | Answer 2 C
      Q6  How many magic items can one creature bond with?               gold            #3 -> #1            Answer 1 C | Answer 2 C
  [embeddinggemma-2] 2 questions; gold chunk outside the baseline top 3 in 1; buried -> promoted in 1; baseline answer fully correct in 1, re-ranked answer fully correct in 2
      Q1  What breaks my focus on a spell?                               gold            #6 -> #2            Answer 1 P | Answer 2 C
      Q6  How many magic items can one creature bond with?               gold            #1 -> #1            Answer 1 C | Answer 2 C
Exceptions & corner cases
  [bge-small] 8 questions; gold chunk outside the baseline top 3 in 2; bu

### 12b. Failure cases in detail: what the baseline got wrong, and why

For each case where the gold chunk was buried and promoted: the chunk the bi-encoder ranked first (the likely **distractor**), the
passage that actually answers the question, and Answer 1 next to Answer 2. The cause line says what the baseline answer did:
*abstained* (said the context does not answer), *misstated a rule*, or was *partly correct* (omitted a clause or added an unsupported detail).

In [20]:
CAUSE = {"A": "abstained because the critical chunk sat outside its top 3",
         "W": "MISSTATED A RULE because the critical chunk sat outside its top 3",
         "P": "partly correct: omitted an essential clause or added an unsupported detail",
         "C": "correct"}
def snippet(cid, pat=None, w=150):
    t = " ".join(chunks[cid]["text"].split())
    if pat:
        m = re.search(pat, t, re.I)
        if m: return t[max(0, m.start() - w // 2): m.end() + w // 2]
    return t[:w]
for emb in EMBS:
    sub = F[(F.embedder == emb) & F.buried_then_promoted]
    print("#" * 100, f"\n{emb}: {len(sub)} detailed cases")
    for _, x in sub.iterrows():
        q = x.question; r = retrieval[emb][q]; gpat = TESTS[x.q - 1][2]; gid = int(x.gold_chunk)
        top1 = r["base_ids"][0]
        a1 = next(y for y in results if y["embedder"] == emb and y["query"] == q and y["pipeline"] == "baseline")
        a2 = next(y for y in results if y["embedder"] == emb and y["query"] == q and y["pipeline"] == "rerank")
        print("-" * 100, f"\nQ{x.q} [{x.type}] {q}\n  Chunk c{int(x.gold_chunk)} moved from Rank #{int(x.rank_baseline)} -> Rank #{int(x.rank_reranked)}")
        print(f"  Baseline's rank-1 chunk c{top1}{' (the gold chunk)' if top1 in r['gold_ids'] else ' (not the answer, a distractor)'}: ...{snippet(top1)}...")
        print(f"  Answering passage c{gid}: ...{snippet(gid, gpat)}...")
        print(f"  gold chunk in the context the LLM saw: Answer 1 = {a1['gold_in_context']}, Answer 2 = {a2['gold_in_context']}")
        print(f"  ANSWER 1 [{x.answer1_verdict}] {CAUSE[x.answer1_verdict]}:\n      {textwrap.fill(a1['answer'], 110, subsequent_indent='      ')}")
        print(f"  ANSWER 2 [{x.answer2_verdict}] {CAUSE[x.answer2_verdict]}:\n      {textwrap.fill(a2['answer'], 110, subsequent_indent='      ')}")

#################################################################################################### 
bge-small: 3 detailed cases
---------------------------------------------------------------------------------------------------- 
Q1 [Keyword trap] What breaks my focus on a spell?
  Chunk c372 moved from Rank #5 -> Rank #2
  Baseline's rank-1 chunk c334 (not the answer, a distractor): ...onto the spell’s magic requires concentration. If your concentration is broken, the spell dissipates without taking effect. For example, if you are co...
  Answering passage c372: ...reath, you make a separate saving throw for each source of damage. • Being incapacitated or killed. You lose concentration on a spell if you are incapacitated or if you die....
  gold chunk in the context the LLM saw: Answer 1 = False, Answer 2 = True
  ANSWER 1 [P] partly correct: omitted an essential clause or added an unsupported detail:
      Taking damage while concentrating on a spell breaks your focus unless you su

## 13. Architecture comparison and the scaling question

The numbers in the sentences below are measured in this notebook.

**Bi-encoder vs cross-encoder.** A bi-encoder (here BGE or EmbeddingGemma 2) turns the question and each chunk into one fixed vector
*independently*, and relevance is the cosine or inner product of two vectors. A cross-encoder (here MS MARCO MiniLM) reads the question
and one chunk together as a single input, so self-attention runs across the tokens of both and a word in the question can attend to a
word in the chunk (for example "see" next to "can't see you"). That joint reading is why it can separate negations, exceptions and
distractors that a single averaged vector blurs together.

**Why not run the cross-encoder on the whole collection first.** The cross-encoder has to read a fresh (question, chunk) pair for every
chunk, and nothing about a chunk can be computed in advance because the score depends on the question. Its cost therefore grows
linearly with the collection, while the bi-encoder embeds every chunk once offline and answers a question with one embedding plus a fast
vector search. The two-stage design spends the cheap model on everything and the expensive model only on a few candidates. Its limit is
that the cross-encoder can only re-order what stage 1 returned: a chunk that is not among the 10 candidates cannot be recovered.

In [21]:
rows_in10 = {emb: sum(retrieval[emb][q]["gold_rank_base"] is not None for _, q, _ in TESTS) for emb in EMBS}
print(f"Cross-encoder on the top 10: {np.mean([e['stage2_ms'] for e in emb_info.values()]):.1f} ms per question.")
print(f"Cross-encoder on all {len(chunks)} chunks: {t_all_ms:.0f} ms per question ({per_chunk_ms:.2f} ms per chunk), "
      f"about {t_all_ms / np.mean([e['stage2_ms'] for e in emb_info.values()]):.0f} times more, and it grows linearly:")
print(scaling.to_string(index=False))
print("\nThe ceiling of the two-stage design: questions whose gold chunk is NOT among the 10 candidates (re-ranking cannot fix these):")
for emb in EMBS:
    miss = [f"Q{n}" for n, (_, q, _) in enumerate(TESTS, 1) if retrieval[emb][q]["gold_rank_base"] is None]
    print(f"  {emb}: {len(miss)} of {len(TESTS)} -> {', '.join(miss)}")

Cross-encoder on the top 10: 17.4 ms per question.
Cross-encoder on all 1504 chunks: 1884 ms per question (1.25 ms per chunk), about 109 times more, and it grows linearly:
 chunks in corpus  cross-encoder over everything (s per question)
             1504                                             1.9
           100000                                           125.3
          1000000                                          1252.9
         10000000                                         12528.9

The ceiling of the two-stage design: questions whose gold chunk is NOT among the 10 candidates (re-ranking cannot fix these):
  bge-small: 3 of 25 -> Q7, Q8, Q23
  embeddinggemma-2: 2 of 25 -> Q5, Q8


## 14. Checklist against the assignment

| Assignment item | Where in this notebook |
|---|---|
| One complex PDF | the D&D 5e SRD, section 1 |
| Pipeline 1: dense embeddings in a vector store, top 10 by cosine / inner product, top 3 to the LLM, Answer 1 | sections 3 and 8 |
| Pipeline 2: same 10 through the cross-encoder `ms-marco-MiniLM-L-6-v2`, re-sort, top 3, identical prompt, Answer 2 | sections 3 and 8 |
| At least two failure queries where the baseline stumbles and re-ranking restores relevance | section 12 (the "RESTORED" cases) |
| Keyword trap / distractor, exceptions and corner cases, negation and constraints, buried chunk promoted from rank 4-10 | sections 12 and 12a, detail in 12b |
| Report 1: architecture comparison | section 13 |
| Report 2: rank changes ("Chunk X moved from Rank #a to #b") and Answer 1 vs Answer 2 (hallucination, omission, misstated rule) | sections 6, 9, 10 and 12b |
| Report 3: latency of the baseline retrieval vs the re-ranking pipeline | section 11 |
| Report 4: why not run the cross-encoder on the whole collection | sections 6 and 13 |